# FedSecureAI — Google Colab GPU Federated Learning Runner

This notebook provides the complete, robust setup to run the **FedSecureAI: FedAvg vs FedProx ($\mu=0.0001$)** comparative suite on **Google Colab with GPU acceleration (T4/V100/A100)**.

### Capabilities
1. **Automatic GPU Detection & cuDNN Acceleration** for GRU, LSTM, and BiLSTM.
2. **Google Drive Integration**: Checkpoints, predictions, and metrics are saved persistently to Google Drive, surviving Colab disconnections.
3. **True Resumption**: Automatically checks `resume_manifest.json` and skips the 12 completed configurations, continuing directly from pending or interrupted models.
4. **Complete Matrix**: Supports both **Non-IID** and **IID** client distributions across HI-Small, LI-Medium, and Combined.


## 1. Verify GPU Hardware and Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Extract Project ZIP Package

In [ ]:
import os
import shutil
from pathlib import Path

# Paths
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/FedSecureAI")
LOCAL_DIR = Path("/content/FedSecureAI")

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(LOCAL_DIR))
print(f"Current working directory: {os.getcwd()}")

# Look for FedSecureAI_colab.zip in Drive or /content
zip_path = DRIVE_PROJECT_DIR / "FedSecureAI_colab.zip"
if not zip_path.exists():
    zip_path = Path("/content/FedSecureAI_colab.zip")

if zip_path.exists():
    print(f"Extracting {zip_path} to {LOCAL_DIR}...")
    !unzip -q -o "{zip_path}" -d "{LOCAL_DIR}"
    print("Extraction complete.")
else:
    print("WARNING: FedSecureAI_colab.zip not found! Please upload it to /content/ or to Google Drive at FedSecureAI/.")


## 4. Install Dependencies

In [ ]:
!pip install -q -r requirements_colab.txt


## 5. Configure Dataset and Persistent Google Drive Storage

In [ ]:
# Check dataset location
DRIVE_DATA = DRIVE_PROJECT_DIR / "data" / "processed" / "ibm_aml"
LOCAL_DATA = LOCAL_DIR / "data" / "processed" / "ibm_aml"

DATA_PATH = DRIVE_DATA if DRIVE_DATA.exists() else LOCAL_DATA
print(f"Using Data Directory: {DATA_PATH}")

# Persistent artifacts path on Drive
PERSISTENT_OUTPUT = DRIVE_PROJECT_DIR / "artifacts" / "fedavg_fedprox_full_comparison"
PERSISTENT_OUTPUT.mkdir(parents=True, exist_ok=True)

# Copy extracted local artifacts/checkpoints to Drive if not already present
LOCAL_OUTPUT = LOCAL_DIR / "artifacts" / "fedavg_fedprox_full_comparison"
if LOCAL_OUTPUT.exists():
    print("Syncing local baseline checkpoints and predictions to Google Drive...")
    shutil.copytree(str(LOCAL_OUTPUT), str(PERSISTENT_OUTPUT), dirs_exist_ok=True)
    print("Artifact sync to Drive complete.")

print(f"Persistent output path: {PERSISTENT_OUTPUT}")


## 6. Verification Dry-Run (Zero Full Training)
Validates GPU execution, cuDNN kernel compilation, dataset readability, and checkpoint integrity.

In [ ]:
!python backend/scripts/run_fedavg_fedprox_comparison_suite.py \
    --data-root "{DATA_PATH}" \
    --output-dir "{PERSISTENT_OUTPUT}" \
    --device auto \
    --verify-only


## 7. Safe First Configuration Test on GPU
Runs a single configuration (HI-Small GRU FedAvg IID) for 5 rounds on GPU (~1-2 minutes) to benchmark throughput.

In [ ]:
!python backend/scripts/run_fedavg_fedprox_comparison_suite.py \
    --data-root "{DATA_PATH}" \
    --output-dir "{PERSISTENT_OUTPUT}" \
    --device auto \
    --datasets HI-Small \
    --strategies iid \
    --architectures gru \
    --algorithms fedavg \
    --max-configs 1


## 8. Full Experiment Execution on GPU (Resumption Enabled)
This will:
1. Skip all 12 completed configurations automatically.
2. Execute remaining configurations (HI-Small IID, Combined IID, LI-Medium IID & Non-IID).
3. Save all checkpoints, predictions, and CSV tables directly to Google Drive persistently.


In [ ]:
!python backend/scripts/run_fedavg_fedprox_comparison_suite.py \
    --data-root "{DATA_PATH}" \
    --output-dir "{PERSISTENT_OUTPUT}" \
    --device auto \
    --strategies both
